In [113]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import os
import re
import math
import numpy as np
from pymorphy2 import MorphAnalyzer
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

MODEL_NAME = "distilbert/distilroberta-base"
MODEL_NAME_TOK = MODEL_NAME
OUTPUT_DIR = "./hw4_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64
EPOCHS_CLS = 7
EPOCHS_MLM = 9

print(f"device: {DEVICE}")
print(torch.__version__)

device: cuda
2.5.1+cu121


In [114]:
morph = MorphAnalyzer()

def clean_text(text, lemmatize=True):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\d+', '<NUM>', text)
    if lemmatize:
        text = ' '.join(morph.parse(w)[0].normal_form for w in text.split())
    return re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)

C:\Users\Igor\anaconda3\envs\torchenv\lib\site-packages\pymorphy2\analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [115]:
df = pd.read_csv("data/train.csv")
with open("data/mlm.txt", "r", encoding="utf-8") as f:
    plain_txt = f.read()
    
df["text"] = df["text"].apply(clean_text)

In [116]:
df.head()

,text,label
0,кровь какой кровь встревожиться,1
1,под нижний подушку,0
2,благодарюс,1
3,когда же этос,1
4,старуха помолчала как бы в раздумье,1


In [117]:
max_len = df["text"].str.len().max()
print(max_len)

63


In [118]:
train_df, valid_df = train_test_split(df, test_size=0.2, stratify=df['label'])

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))
dataset = DatasetDict({"train": train_ds, "validation": valid_ds})
full_dataset = DatasetDict({"train": Dataset.from_pandas(df.reset_index(drop=True))})

print(f"train_df.shape: {train_df.shape}, valid_df.shape: {valid_df.shape}")

train_df.shape: (4512, 2), valid_df.shape: (1128, 2)


In [119]:
train_df.head()

,text,label
5314,худое истощенное желтоватый лицо он быть,0
2692,в семь часов кажется,0
4358,ростов пожать плечами как будто говоря,0
998,ежели он быть браниться я,0
711,послушайте князь сказать она,0


In [120]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_TOK, use_fast=True)

In [121]:
def preprocess_classification(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(preprocess_classification, batched=True)
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in ("input_ids","attention_mask","label")])
tokenized.set_format(type="torch")

full_tokenized = full_dataset.map(preprocess_classification, batched=True)
full_tokenized = full_tokenized.remove_columns([c for c in full_tokenized["train"].column_names if c not in ("input_ids","attention_mask","label")])
full_tokenized.set_format(type="torch")

Map: 100%|██████████| 5640/5640 [00:00<00:00, 31415.30 examples/s]


In [122]:
tokenized

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 4512
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1128
    })
})

In [123]:
full_tokenized

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 5640
    })
})

## Baseline classifier

In [124]:
def compute_metrics_cls(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "cm": confusion_matrix(labels, preds).tolist()
    }

In [125]:
model_cls = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_cls.to(DEVICE)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
           

In [126]:
training_args_baseline = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "cls_baseline"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS_CLS,
    eval_strategy="epoch",
    learning_rate=2e-5,
    fp16=torch.cuda.is_available()
)

trainer_cls = Trainer(
    model=model_cls,
    args=training_args_baseline,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics_cls
)

In [127]:
trainer_cls.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Cm
1,No log,0.654048,0.619681,0.648649,"[[303, 230], [199, 396]]"
2,No log,0.613332,0.663121,0.739726,"[[208, 325], [55, 540]]"
3,No log,0.650435,0.634752,0.738579,"[[134, 399], [13, 582]]"
4,No log,0.594425,0.685284,0.717131,"[[323, 210], [145, 450]]"
5,No log,0.615554,0.673759,0.752355,"[[201, 332], [36, 559]]"
6,No log,0.588269,0.700355,0.735938,"[[319, 214], [124, 471]]"
7,No log,0.589234,0.712766,0.754545,"[[306, 227], [97, 498]]"


TrainOutput(global_step=497, training_loss=0.5767119808695925, metrics={'train_runtime': 852.4725, 'train_samples_per_second': 37.05, 'train_steps_per_second': 0.583, 'total_flos': 1045962579787776.0, 'train_loss': 0.5767119808695925, 'epoch': 7.0})

In [128]:
print("Baseline metrics:")
baseline_metrics = trainer_cls.evaluate()
print(baseline_metrics)

Baseline metrics:


{'eval_loss': 0.589233934879303, 'eval_accuracy': 0.7127659574468085, 'eval_f1': 0.7545454545454545, 'eval_cm': [[306, 227], [97, 498]], 'eval_runtime': 0.6699, 'eval_samples_per_second': 1683.947, 'eval_steps_per_second': 26.871, 'epoch': 7.0}


## Pretraining model with MLM

In [129]:
with open("./data/mlm.txt", "r", encoding="utf-8") as f:
    mlm_raw = f.read()
    
mlm_raw = clean_text(mlm_raw)

In [130]:
enc = tokenizer(mlm_raw, return_tensors="pt", truncation=False) # type(enc) = BatchEncoding
input_ids = enc["input_ids"][0].tolist()
chunk_size = 200  
chunks = [tokenizer.decode(input_ids[i:i+chunk_size], skip_special_tokens=True, clean_up_tokenization_spaces=True)
          for i in range(0, len(input_ids), chunk_size)]
mlm_lines = [c for c in chunks if c.strip()]

Token indices sequence length is longer than the specified maximum sequence length for this model (2458799 > 512). Running this sequence through the model will result in indexing errors


In [131]:
len(mlm_lines)

12294

In [132]:
mlm_lines[0]

' кровь какой кровь  встревожиться пульхерия александровна  под нижний подушку  благодарюс  когда же этос старуха помолчала как бы в раздумье потом отступить в сторона и указывать на дверь'

In [133]:
len(input_ids)

2458799

In [134]:
type(enc)

transformers.tokenization_utils_base.BatchEncoding

In [135]:
mlm_dataset = Dataset.from_pandas(pd.DataFrame({"text": mlm_lines})).train_test_split(test_size=0.1)
mlm_dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 11064
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1230
    })
})

In [136]:
def tokenize_mlm(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_mlm = mlm_dataset.map(tokenize_mlm, batched=True, remove_columns=["text"])
tokenized_mlm.set_format(type="torch")

Map: 100%|██████████| 1230/1230 [00:00<00:00, 15908.63 examples/s]


In [137]:
tokenized_mlm

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 11064
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1230
    })
})

In [138]:
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.to(DEVICE)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.2)

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


#### Metrics before MLM

In [139]:
training_args_mlm = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "mlm_pretrain"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS_MLM,
    eval_strategy="epoch",
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
)

trainer_mlm = Trainer(
    model=mlm_model,
    args=training_args_mlm,
    train_dataset=tokenized_mlm["train"],
    eval_dataset=tokenized_mlm["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

eval_before = trainer_mlm.evaluate()
loss_before = eval_before.get("eval_loss", None)
ppl_before = math.exp(loss_before) if loss_before is not None else None
print(f"MLM before: loss={loss_before}, perplexity={ppl_before}")

C:\Users\Igor\AppData\Local\Temp\ipykernel_47432\661995806.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_mlm = Trainer(


MLM before: loss=1.3019263744354248, perplexity=3.676371919519955


#### Metrics after MLM

In [140]:
trainer_mlm.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,0.963904,0.001000
2,No log,0.874049,0.001000
3,1.033300,0.812885,0.001000
4,1.033300,0.777415,0.001000
5,1.033300,0.741485,0.001000
6,0.855300,0.721670,0.001000
7,0.855300,0.715550,0.001000
8,0.855300,0.708866,0.001000
9,0.798100,0.712028,0.001000


TrainOutput(global_step=1557, training_loss=0.891529240053896, metrics={'train_runtime': 8567.7219, 'train_samples_per_second': 11.622, 'train_steps_per_second': 0.182, 'total_flos': 5232122986117104.0, 'train_loss': 0.891529240053896, 'epoch': 9.0})

In [141]:
mlm_ckpt = os.path.join(OUTPUT_DIR, "mlm_pretrained")
trainer_mlm.save_model(mlm_ckpt)

eval_after = trainer_mlm.evaluate()
loss_after = eval_after.get("eval_loss", None)
ppl_after = math.exp(loss_after) if loss_after is not None else None
print(f"MLM after: loss={loss_after}, perplexity={ppl_after}")

MLM after: loss=0.7051594853401184, perplexity=2.0241694845394593


## Improve baseline classifier

In [142]:
model_cls_from_mlm = AutoModelForSequenceClassification.from_pretrained(mlm_ckpt, num_labels=2)
model_cls_from_mlm.to(DEVICE)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ./hw4_output\mlm_pretrained and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
           

In [143]:
training_args_cls2 = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "cls_from_mlm"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS_CLS,
    eval_strategy="epoch",
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
)

trainer_cls2 = Trainer(
    model=model_cls_from_mlm,
    args=training_args_cls2,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics_cls,
)

In [144]:
trainer_cls2.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Cm
1,No log,0.644194,0.634752,0.626134,"[[371, 162], [250, 345]]"
2,No log,0.597398,0.678191,0.716185,"[[307, 226], [137, 458]]"
3,No log,0.563400,0.693262,0.747813,"[[269, 264], [82, 513]]"
4,No log,0.576096,0.695035,0.707980,"[[367, 166], [178, 417]]"
5,No log,0.548486,0.710993,0.771388,"[[252, 281], [45, 550]]"
6,No log,0.535235,0.731383,0.763466,"[[336, 197], [106, 489]]"
7,No log,0.545791,0.731383,0.764934,"[[332, 201], [102, 493]]"


TrainOutput(global_step=497, training_loss=0.5318440182108275, metrics={'train_runtime': 169.7906, 'train_samples_per_second': 186.017, 'train_steps_per_second': 2.927, 'total_flos': 1045962579787776.0, 'train_loss': 0.5318440182108275, 'epoch': 7.0})

In [145]:
res_cls2 = trainer_cls2.evaluate()
print("Classifier from MLM eval:", res_cls2)

Classifier from MLM eval: {'eval_loss': 0.5457905530929565, 'eval_accuracy': 0.7313829787234043, 'eval_f1': 0.764934057408844, 'eval_cm': [[332, 201], [102, 493]], 'eval_runtime': 2.4236, 'eval_samples_per_second': 465.422, 'eval_steps_per_second': 7.427, 'epoch': 7.0}


#### Retrain model with full dataset

In [146]:
training_args_cls2_full = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "cls_from_mlm"),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS_CLS,
    learning_rate=1e-5,
    fp16=torch.cuda.is_available(),
)

trainer_cls2_full = Trainer(
    model=model_cls_from_mlm,
    args=training_args_cls2_full,
    train_dataset=full_tokenized["train"],
    eval_dataset=None,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_cls,
)

In [147]:
trainer_cls2_full.train()

Step,Training Loss
500,0.422200


TrainOutput(global_step=623, training_loss=0.41384979541956135, metrics={'train_runtime': 271.4681, 'train_samples_per_second': 145.431, 'train_steps_per_second': 2.295, 'total_flos': 1307453224734720.0, 'train_loss': 0.41384979541956135, 'epoch': 7.0})

## Metrics comparison

Метрики на обучении после MLM оказались выше, чем на обучении без MLM. Accuracy в среднем на 2%.


## Predict on test dataset

In [148]:
test_df = pd.read_csv("./data/submission.csv")
test_df["text"] = test_df["text"].apply(clean_text)
ds = Dataset.from_dict({"text": test_df["text"].tolist()})

tokenized_ds = ds.map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=128),
    batched=True
)
tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map: 100%|██████████| 1440/1440 [00:00<00:00, 18007.69 examples/s]


In [149]:
tokenized_ds

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1440
})

In [150]:
pred_output = trainer_cls2_full.predict(tokenized_ds)
preds = np.argmax(pred_output.predictions, axis=-1)

test_df["label"] = preds
test_df.to_csv("./data/submission_with_preds.csv", index=False)